# Leer lugares y personajes

In [1]:
import pandas as pd

In [2]:
ruta_lugares = "../data/external/terminos/Lugares literarios.xlsx"
df_lugares = pd.read_excel(ruta_lugares)
df_lugares.head()

,Autor,Obra principal,Lugar ficticio,Descripción breve del lugar
0,J.K. Rowling,Harry Potter,Callejón Diagon (Diagon Alley),La calle mágica oculta en Londres donde los ma...
1,A. A. Milne,Winnie the Pooh,Bosque de los Cien Acres (Hundred Acre Wood),"El hogar de Pooh, Piglet, Tigger y sus amigos,..."
2,Aldous Huxley,Un mundo feliz,El Estado Mundial (World State),"La sociedad distópica global de la novela, bas..."
3,C. S. Lewis,Las Crónicas de Narnia,Narnia,Un reino mágico al que se accede a través de u...
4,Carlo Collodi,Las aventuras de Pinocho,País de los juguetes (o Juguetelandia),La isla de la tentación donde los niños que no...


In [3]:
ruta_personajes = "../data/external/terminos/Personajes literarios.xlsx"
df_personajes = pd.read_excel(ruta_personajes)
df_personajes.head()

,Autor,Obra principal,Personaje ficticio,Rol principal / característica
0,A. A. Milne,Winnie the Pooh,Winnie the Pooh,"El oso tierno y despistado, amante de la miel."
1,Agatha Christie,Novelas de Poirot,Hércules Poirot,"El detective belga, famoso por sus ""pequeñas c..."
2,Agatha Christie,Novelas de Miss Marple,Miss Marple,La anciana observadora que resuelve crímenes e...
3,Albert Camus,El Extranjero,Meursault,"El protagonista indiferente, símbolo del absur..."
4,Albert Camus,La peste,Dr. Bernard Rieux,"El médico que lucha contra la epidemia, símbol..."


In [8]:
len(df_lugares), len(df_personajes)

(53, 125)

In [4]:
textos = pd.read_parquet("../data/processed/chunks.parquet")
textos.head()

,id_doc,autor_doc,fecha_doc,diario_doc,titulo_doc,chunk_id,texto_chunk
0,1,Gonzalo Hernández,2018-01-01,El Espectador,Fajardo: para nada tibio,0,"La Coalición Colombia Partido Alianza Verde, P..."
1,1,Gonzalo Hernández,2018-01-01,El Espectador,Fajardo: para nada tibio,1,"al mismo tiempo lo exponen, en ciertas ocasion..."
2,1,Gonzalo Hernández,2018-01-01,El Espectador,Fajardo: para nada tibio,2,los acuerdos con las Farc. Anunció que no prom...
3,1,Gonzalo Hernández,2018-01-01,El Espectador,Fajardo: para nada tibio,3,moratoria en la explotación tipo fracking. Y f...
4,2,Eduardo Barajas Sandoval,2018-01-01,El Espectador,Macedonia de Norte,0,Las interpretaciones de la historia sirven com...


# Búsqueda exacta

In [10]:
import pandas as pd
import re
from unidecode import unidecode
from rapidfuzz import fuzz
import time

# ─────────────────────────────────────────────
# 1. NORMALIZACIÓN
# ─────────────────────────────────────────────

def normalizar(texto):
    if pd.isna(texto):
        return ""
    t = str(texto).lower()
    t = unidecode(t)
    t = re.sub(r'[^\w\s]', ' ', t)
    t = re.sub(r'\s+', ' ', t).strip()
    return t


TITULOS = r'\b(dr|dra|doctor|doctora|mr|mrs|miss|sr|sra|don|dona|lord|lady|sir|prof|profesor|profesora)\b\.?\s*'

def generar_variantes(nombre):
    """Genera 2-4 variantes útiles sin explotar en cantidad"""
    nombre_norm = normalizar(nombre)
    variantes = {nombre_norm}

    # Sin título al inicio
    sin_titulo = re.sub(TITULOS, '', nombre_norm, flags=re.IGNORECASE).strip()
    if sin_titulo:
        variantes.add(sin_titulo)

    # Solo primer nombre
    partes = nombre_norm.split()
    if len(partes) > 1:
        variantes.add(partes[0])   # primer token
        variantes.add(partes[-1])  # último token (apellido)

    # Eliminar variantes muy cortas (ruido)
    return [v for v in variantes if len(v) >= 4]


# ─────────────────────────────────────────────
# 2. ÍNDICE DE BÚSQUEDA (compilar regex una sola vez)
# ─────────────────────────────────────────────

def construir_indice(df_entidades, col_entidad):
    """
    Devuelve lista de dicts:
    {'original': str, 'patron': re.Pattern, 'autor': str, 'obra': str}
    """
    indice = []
    for _, row in df_entidades.iterrows():
        nombre_original = str(row[col_entidad])
        variantes = generar_variantes(nombre_original)

        # Ordenar por longitud desc para que el match más largo gane
        variantes.sort(key=len, reverse=True)

        # Compilar un único patrón alternado por entidad
        patron_str = '|'.join(re.escape(v) for v in variantes)
        patron = re.compile(rf'\b({patron_str})\b')

        indice.append({
            'original': nombre_original,
            'patron': patron,
            'variantes': variantes,
            'autor': row.get('Autor', ''),
            'obra': row.get('Obra principal', ''),
        })
    return indice


# ─────────────────────────────────────────────
# 3. BÚSQUEDA VECTORIZADA (sin loop de ventanas)
# ─────────────────────────────────────────────

def buscar_entidades_rapido(df_textos, df_entidades, col_entidad,
                            nombre_col_resultado, umbral_fuzzy=88):
    """
    Estrategia de dos pasos:
      1. Regex exacto sobre texto normalizado  →  O(n * e)  rápido
      2. Fuzzy solo si el nombre tiene > 1 palabra (para nombres compuestos)
         y el texto contiene el primer token (filtro previo barato)
    """
    indice = construir_indice(df_entidades, col_entidad)

    # Pre-normalizar todos los textos de una vez
    textos_norm = df_textos['texto_chunk'].apply(normalizar).tolist()
    textos_orig = df_textos['texto_chunk'].tolist()
    ids_doc    = df_textos['id_doc'].tolist()
    ids_chunk  = df_textos['chunk_id'].tolist()

    resultados = []
    total = len(textos_norm)
    t0 = time.time()

    for i, (texto_norm, texto_orig) in enumerate(zip(textos_norm, textos_orig)):

        # ── progreso cada 5000 chunks ──
        if i % 5000 == 0 and i > 0:
            elapsed = time.time() - t0
            velocidad = i / elapsed
            restante = (total - i) / velocidad
            print(f"  {i}/{total} chunks  |  {elapsed/60:.1f} min transcurridos  |  ~{restante/60:.1f} min restantes")

        for ent in indice:

            encontrado = False

            # ── PASO 1: regex exacto (muy rápido) ──
            if ent['patron'].search(texto_norm):
                match_txt = ent['patron'].search(texto_norm).group(0)
                resultados.append({
                    'id_doc':              ids_doc[i],
                    'chunk_id':            ids_chunk[i],
                    'texto_chunk':         texto_orig,
                    nombre_col_resultado:  ent['original'],
                    'similitud':           100,
                    'metodo':             'exacto',
                    'autor_ref':           ent['autor'],
                    'obra_ref':            ent['obra'],
                })
                encontrado = True

            # ── PASO 2: fuzzy solo para nombres compuestos (≥2 tokens) ──
            # y solo si al menos UN token del nombre aparece en el texto
            if not encontrado:
                variante_larga = ent['variantes'][0]  # la más larga va primero
                tokens = variante_larga.split()

                if len(tokens) >= 2:
                    # Filtro barato: ¿está el primer o último token?
                    if tokens[0] in texto_norm or tokens[-1] in texto_norm:
                        sim = fuzz.partial_ratio(variante_larga, texto_norm)
                        if sim >= umbral_fuzzy:
                            resultados.append({
                                'id_doc':              ids_doc[i],
                                'chunk_id':            ids_chunk[i],
                                'texto_chunk':         texto_orig,
                                nombre_col_resultado:  ent['original'],
                                'similitud':           sim,
                                'metodo':             'fuzzy',
                                'autor_ref':           ent['autor'],
                                'obra_ref':            ent['obra'],
                            })

    elapsed_total = time.time() - t0
    print(f"\n✅ Finalizado en {elapsed_total/60:.1f} min — {len(resultados)} coincidencias encontradas")
    return pd.DataFrame(resultados)


# ─────────────────────────────────────────────
# 4. USO
# ─────────────────────────────────────────────

if __name__ == "__main__":

    # Cargar datos
    ruta_lugares    = "../data/external/terminos/Lugares literarios.xlsx"
    ruta_personajes = "../data/external/terminos/Personajes literarios.xlsx"

    df_lugares    = pd.read_excel(ruta_lugares)
    df_personajes = pd.read_excel(ruta_personajes)
    textos        = pd.read_parquet("../data/processed/chunks.parquet")

    # ── Lugares ──────────────────────────────
    print("Buscando lugares literarios...")
    df_lugares_encontrados = buscar_entidades_rapido(
        textos,
        df_lugares,
        col_entidad='Lugar ficticio',
        nombre_col_resultado='lugar_literario',
        umbral_fuzzy=88,
    )
    # Forzar todas las columnas de texto a string antes de guardar
    for col in ['lugar_literario', 'metodo', 'autor_ref', 'obra_ref', 'variante_encontrada'] :
        if col in df_lugares_encontrados.columns:
            df_lugares_encontrados[col] = df_lugares_encontrados[col].astype(str)

    df_lugares_encontrados.to_parquet("../data/processed/lugares_encontrados.parquet", index=False)
    print(df_lugares_encontrados.head())

    # ── Personajes ───────────────────────────
    print("\nBuscando personajes literarios...")
    df_personajes_encontrados = buscar_entidades_rapido(
        textos,
        df_personajes,
        col_entidad='Personaje ficticio',
        nombre_col_resultado='personaje_literario',
        umbral_fuzzy=88,
    )

    # Forzar todas las columnas de texto a string antes de guardar
    for col in ['personaje_literario', 'metodo', 'autor_ref', 'obra_ref']:
        if col in df_personajes_encontrados.columns:
            df_personajes_encontrados[col] = df_personajes_encontrados[col].astype(str)

    df_personajes_encontrados.to_parquet("../data/processed/personajes_encontrados.parquet", index=False)
    print(df_personajes_encontrados.head())

Buscando lugares literarios...
  5000/62651 chunks  |  0.1 min transcurridos  |  ~1.7 min restantes
  10000/62651 chunks  |  0.3 min transcurridos  |  ~1.5 min restantes
  15000/62651 chunks  |  0.4 min transcurridos  |  ~1.4 min restantes
  20000/62651 chunks  |  0.6 min transcurridos  |  ~1.4 min restantes
  25000/62651 chunks  |  1.0 min transcurridos  |  ~1.4 min restantes
  30000/62651 chunks  |  1.3 min transcurridos  |  ~1.4 min restantes
  35000/62651 chunks  |  1.5 min transcurridos  |  ~1.2 min restantes
  40000/62651 chunks  |  1.8 min transcurridos  |  ~1.0 min restantes
  45000/62651 chunks  |  2.1 min transcurridos  |  ~0.8 min restantes
  50000/62651 chunks  |  2.4 min transcurridos  |  ~0.6 min restantes
  55000/62651 chunks  |  2.7 min transcurridos  |  ~0.4 min restantes
  60000/62651 chunks  |  3.0 min transcurridos  |  ~0.1 min restantes

✅ Finalizado en 3.1 min — 62188 coincidencias encontradas
   id_doc  chunk_id                                        texto_chunk 

In [12]:
with pd.ExcelWriter("../data/processed/coincidencias_literarias.xlsx") as writer:
    df_lugares_encontrados.to_excel(writer, sheet_name="Lugares", index=False)
    df_personajes_encontrados.to_excel(writer, sheet_name="Personajes", index=False)

print("✅ Excel guardado en ../data/processed/coincidencias_literarias.xlsx")

✅ Excel guardado en ../data/processed/coincidencias_literarias.xlsx
